# CEO Supervisor — MLflow Experiment Setup

Run this notebook **once** after the pipeline has completed to configure the MLflow experiment
for ongoing quality monitoring. It is **not** part of the deployment pipeline.

## What this notebook does

| Step | Action |
|------|--------|
| 1 | Locate the MLflow experiment created by `ceo_evaluation.ipynb` |
| 2 | Create a **persistent MLflow-managed eval dataset** in Unity Catalog |
| 3 | **Sessions**: already wired — the dashboard sends `Databricks-Source-Conversation-Id` per chat session, so the MLflow Sessions tab populates automatically |
| 4 | Register and start **production monitoring scorers** (same judges as `ceo_evaluation.ipynb`) |


In [ ]:
%pip install --upgrade mlflow databricks-sdk 'mlflow[databricks]'
dbutils.library.restartPython()

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import sys, json as _json
sys.path.append("../../utils")

import mlflow
import mlflow.genai
import mlflow.genai.datasets
from mlflow import MlflowClient
from databricks.sdk import WorkspaceClient

CATALOG = dbutils.widgets.get("CATALOG")

_w = WorkspaceClient()
_current_user = spark.sql("SELECT current_user()").collect()[0][0]


def _uc_state_endpoint(catalog: str) -> str:
    """Read the MAS endpoint name written by ceo_supervisor.ipynb into uc_state."""
    try:
        rows = spark.sql(f"""
            SELECT resource_data
            FROM {catalog}._internal_state.resources
            WHERE resource_type = 'multi_agent_supervisors'
            ORDER BY created_at DESC
            LIMIT 1
        """).collect()
        if rows:
            data = _json.loads(rows[0].resource_data)
            return data.get("endpoint_name", "")
    except Exception:
        pass
    return ""


def _mas_api_endpoint(tile_name: str) -> str:
    """Check the MAS API directly — handles mas-{id}-endpoint naming."""
    try:
        params = {}
        while True:
            resp = _w.api_client.do("GET", "/api/2.0/multi-agent-supervisors", query=params)
            for item in resp.get("multi_agent_supervisors", []):
                mas = item.get("multi_agent_supervisor", item)
                if mas.get("name") == tile_name:
                    ep = mas.get("endpoint_name", "")
                    if ep:
                        return ep
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception:
        pass
    # Tiles API fallback
    try:
        params = {}
        while True:
            resp = _w.api_client.do("GET", "/api/2.0/tiles", query=params)
            for tile in resp.get("tiles", []):
                if tile.get("name") == tile_name:
                    ep = tile.get("serving_endpoint_name", "")
                    if ep:
                        return ep
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception:
        pass
    return ""


_supervisor_tile_name = f"{CATALOG}-ceo-supervisor"

SUPERVISOR_ENDPOINT = _uc_state_endpoint(CATALOG)
if not SUPERVISOR_ENDPOINT:
    SUPERVISOR_ENDPOINT = _mas_api_endpoint(_supervisor_tile_name)

if not SUPERVISOR_ENDPOINT:
    raise ValueError(
        f"Could not resolve supervisor endpoint for '{_supervisor_tile_name}'. "
        "Run the CEO Supervisor stage first."
    )

_mas_id = SUPERVISOR_ENDPOINT.replace("-endpoint", "")

_eval_exp_name    = f"/Users/{_current_user}/{_mas_id}-dev-experiment"
_serving_exp_name = f"/Serving/{SUPERVISOR_ENDPOINT}"

mlflow.set_tracking_uri("databricks")
client = MlflowClient()

# Dashboard traces are auto-logged to /Serving/{endpoint} by Databricks.
# Register scorers there so they actually run on production traffic.
# Fall back to the eval experiment if the serving one doesn't exist yet.
_serving_exp = mlflow.get_experiment_by_name(_serving_exp_name)
_eval_exp    = mlflow.get_experiment_by_name(_eval_exp_name)

if _serving_exp:
    EXP_ID   = _serving_exp.experiment_id
    EXP_NAME = _serving_exp_name
    print(f"✅ Using serving experiment (production traces): {EXP_NAME}")
elif _eval_exp:
    EXP_ID   = _eval_exp.experiment_id
    EXP_NAME = _eval_exp_name
    print(f"ℹ️  Serving experiment not created yet — using eval experiment.")
    print(f"   Send a message through the dashboard, then re-run to register scorers on production traces.")
else:
    EXP_ID   = mlflow.create_experiment(_eval_exp_name)
    EXP_NAME = _eval_exp_name
    print(f"✅ Created eval experiment: {EXP_NAME}")

print(f"\nCatalog:    {CATALOG}")
print(f"Endpoint:   {SUPERVISOR_ENDPOINT}")
print(f"Experiment: {EXP_NAME} → id={EXP_ID}")


## 1. Create MLflow-Managed Evaluation Dataset

This creates a **persistent, versioned dataset** in Unity Catalog backed by the same curated
evaluation examples used in `ceo_evaluation.ipynb`.

Unlike `mlflow.log_input()` (which attaches a snapshot to a single run), an MLflow-managed
dataset lives in UC, can be reused across runs, appended to from production traces, and
referenced by name from any notebook or job.

In [ ]:
# ── Curated evaluation records — mirrors EVAL_DATASET in ceo_evaluation.ipynb ─
EVAL_RECORDS = [
    # ── Revenue ──────────────────────────────────────────────────────────────
    {
        "inputs": {"question": "Which location has the highest order cancellation rate right now?", "expected_agent": "revenue"},
        "expectations": {
            "expected_agent": "revenue",
            "guidelines": [
                "Must name exactly one specific location as having the highest rate — not a list.",
                "Must include a numeric cancellation rate (percentage).",
                "Must not say it cannot access the data or ask for clarification.",
            ],
        },
    },
    {
        "inputs": {"question": "How does revenue compare across our locations this week?", "expected_agent": "revenue"},
        "expectations": {
            "expected_agent": "revenue",
            "guidelines": [
                "Must provide revenue figures or a ranking for multiple locations.",
                "Must reference the current week as the time period.",
                "Must not return a generic description without actual numbers.",
            ],
        },
    },
    {
        "inputs": {"question": "Which brand is generating the most revenue right now?", "expected_agent": "revenue"},
        "expectations": {
            "expected_agent": "revenue",
            "guidelines": [
                "Must name a specific brand (not a location).",
                "Must include a revenue figure or ranking.",
                "Must distinguish between brand-level and location-level performance.",
            ],
        },
    },
    # ── Operations ───────────────────────────────────────────────────────────
    {
        "inputs": {"question": "Which location needs the most operational attention right now?", "expected_agent": "operations"},
        "expectations": {
            "expected_agent": "operations",
            "guidelines": [
                "Must name one specific location with the highest risk.",
                "Must justify with at least two operational metrics (e.g. complaint rate, cancel rate, food safety).",
                "Must not give a generic answer — must reference actual data from the system.",
            ],
        },
    },
    # ── Inspection ───────────────────────────────────────────────────────────
    {
        "inputs": {"question": "What happened during the Chicago food safety inspection? Were there critical violations?", "expected_agent": "inspection"},
        "expectations": {
            "expected_agent": "inspection",
            "guidelines": [
                "Must reference a specific inspection report by date or ID.",
                "Must state the inspection score and/or grade.",
                "Must explicitly state whether critical violations were found.",
                "Must cite at least one specific violation if they exist.",
            ],
        },
    },
    # ── Legal ─────────────────────────────────────────────────────────────────
    {
        "inputs": {"question": "Do we have any active high-risk legal cases? What is the total financial exposure?", "expected_agent": "legal"},
        "expectations": {
            "expected_agent": "legal",
            "guidelines": [
                "Must confirm whether active cases exist — cannot hedge or say data is unavailable.",
                "Must cite at least one specific case number (format: CK-XX-XXXX).",
                "Must include a risk classification (HIGH / MEDIUM / LOW) for at least one case.",
                "Must state a financial exposure amount.",
                "Must include a reminder to involve legal counsel.",
            ],
        },
    },
    {
        "inputs": {"question": "Which location has the most active legal cases?", "expected_agent": "legal"},
        "expectations": {
            "expected_agent": "legal",
            "guidelines": [
                "Must name a specific location — not 'I don't know' or a list of all locations.",
                "Must provide a count of active cases for at least the top location.",
                "Must distinguish active cases from settled or dismissed ones.",
            ],
        },
    },
    # ── Regulatory ───────────────────────────────────────────────────────────
    {
        "inputs": {"question": "Are there any permits or regulatory certificates expiring in the next 60 days?", "expected_agent": "regulatory"},
        "expectations": {
            "expected_agent": "regulatory",
            "guidelines": [
                "Must list specific document IDs or permit names — not generic descriptions.",
                "Must include expiry dates for each flagged document.",
                "Must specify which location each document applies to.",
                "If nothing is expiring, must say so explicitly with supporting evidence.",
            ],
        },
    },
    # ── Consultancy ──────────────────────────────────────────────────────────
    {
        "inputs": {"question": "What do our consultants recommend as the top AI investments for the next 90 days?", "expected_agent": "consultancy"},
        "expectations": {
            "expected_agent": "consultancy",
            "guidelines": [
                "Must reference a specific consulting report or firm by name.",
                "Must include at least one concrete recommendation (not just 'invest in AI').",
                "Must include a financial metric — ROI estimate, cost, or projected saving.",
                "Must frame the answer in the 90-day horizon.",
            ],
        },
    },
    # ── Audit ─────────────────────────────────────────────────────────────────
    {
        "inputs": {"question": "What were the most significant audit findings this quarter?", "expected_agent": "audits"},
        "expectations": {
            "expected_agent": "audits",
            "guidelines": [
                "Must cite the auditing firm and the audit period covered.",
                "Must classify findings by severity (Critical / Significant / Minor / Informational).",
                "Must state remediation status for at least one finding.",
                "Must not conflate audit findings with food safety inspection findings.",
            ],
        },
    },
    # ── Cross-domain synthesis ────────────────────────────────────────────────
    {
        "inputs": {"question": "Give me a board deck summary: revenue performance, top operational risk, legal exposure, and one strategic recommendation.", "expected_agent": "multi"},
        "expectations": {
            "expected_agent": "multi",
            "guidelines": [
                "Must explicitly address all four domains: revenue, operations, legal, and strategy.",
                "Must include at least one concrete number or data point per domain.",
                "Must not refuse or hedge on any of the four domains.",
                "Response must be structured — not a single unbroken paragraph.",
            ],
        },
    },
]

# ── Create or update the MLflow-managed dataset in Unity Catalog ──────────────
UC_DATASET_TABLE = f"{CATALOG}.evaluations.ceo_supervisor_eval_dataset"

# Drop any previously created table — an old run may have written guidelines as
# ARRAY<STRING>, which Arrow cannot serialize.  Recreate from scratch each run.
spark.sql(f"DROP TABLE IF EXISTS {UC_DATASET_TABLE}")

eval_dataset = mlflow.genai.datasets.create_dataset(uc_table_name=UC_DATASET_TABLE)
print(f"✅ Created dataset: {UC_DATASET_TABLE}")

# merge_records() writes via Arrow/Spark — ARRAY<STRING> fails serialization.
# Serialize every list-type field to a JSON string before writing.
def _serialize_record(rec: dict) -> dict:
    out = {}
    for k, v in rec.items():
        if isinstance(v, dict):
            out[k] = _serialize_record(v)
        elif isinstance(v, list):
            out[k] = _json.dumps(v)
        else:
            out[k] = v
    return out

eval_dataset.merge_records([_serialize_record(r) for r in EVAL_RECORDS])
print(f"✅ Dataset updated — {len(EVAL_RECORDS)} curated records")

# Log the dataset as input to a run so it appears in the MLflow UI under the experiment.
mlflow.set_experiment(EXP_NAME)
with mlflow.start_run(run_name="register_eval_dataset"):
    mlflow.log_input(
        mlflow.data.from_spark(spark.table(UC_DATASET_TABLE), table_name=UC_DATASET_TABLE),
        context="eval",
    )
print(f"✅ Dataset linked to experiment in MLflow UI")
print(f"   UC table: {UC_DATASET_TABLE}")
print(f"   Use in evaluate(): mlflow.genai.evaluate(data=eval_dataset, ...)")


## 2. Sessions — Already Wired

Sessions are **already enabled** in the CEO Dashboard:

- The backend (`main.py`) passes `Databricks-Source-Conversation-Id: {session_id}` in every
  request to the MAS endpoint — one session ID per Lakebase chat session.
- MLflow automatically groups all traces from the same session together under the
  **Sessions tab** in the Experiments UI.
- No extra configuration needed — the tab appears as soon as the first traced conversation arrives.

The cell below sets an experiment tag on the serving experiment to ensure the Sessions view
is surfaced in the UI even before the first trace arrives.

In [ ]:
client.set_experiment_tag(EXP_ID, "mlflow.chat", "true")
print(f"✅ Sessions enabled: {EXP_NAME}")
if _eval_exp and EXP_ID != _eval_exp.experiment_id:
    client.set_experiment_tag(_eval_exp.experiment_id, "mlflow.chat", "true")
    print(f"✅ Sessions enabled: {_eval_exp_name}")


## 3. Production Monitoring Scorers

All scorers work on question + response only — no expected outputs needed for production traces:

| Scorer | What it checks |
|--------|----------------|
| `Safety` | Harmful / inappropriate content |
| `RelevanceToQuery` | Response directly answers the question asked |
| `ceo_quality` (Guidelines) | Response contains specific data; not a hedge or refusal |
| `routing_accuracy` | Response contains domain-specific vocabulary from a sub-agent |
| `cites_specific_data` | Response contains numbers, dates, IDs rather than vague hedging |

In [ ]:
import re as _re
from mlflow.genai.scorers import (
    scorer, Safety, Guidelines, RelevanceToQuery,
    ScorerSamplingConfig, list_scorers,
)
from mlflow.entities import Feedback

_AGENT_SIGNALS = {
    "revenue":     ["revenue", "cancellation rate", "order count", "orders placed", "sales", "avg order", "weekly", "total orders"],
    "operations":  ["complaint rate", "kitchen", "operational", "throughput", "busiest", "food safety grade", "cancel rate"],
    "inspection":  ["inspection report", "violation", "inspector", "corrective action", "grade", "score", "health permit"],
    "legal":       ["case no", "ck-", "risk level", "amount at stake", "counsel", "litigation", "plaintiff", "high risk"],
    "regulatory":  ["permit", "certificate", "expiry", "regulatory", "fda", "zoning", "issuing authority"],
    "audits":      ["audit", "auditor", "finding", "pwc", "deloitte", "kpmg", "significant", "critical finding"],
    "consultancy": ["consultant", "roi", "recommend", "strategy", "mckinsey", "phase 1", "investment"],
    "multi":       ["revenue", "legal", "audit", "operational"],
}


@scorer
def routing_accuracy(inputs: dict, outputs: str, expectations: dict = None) -> Feedback:
    """Checks whether the response contains domain-specific vocabulary from any sub-agent.

    Works with or without expected_agent:
    - With expected_agent (eval runs): checks vocabulary match for that specific domain.
    - Without expected_agent (production traces): checks whether the response contains
      strong vocabulary signals from at least one domain — i.e. the agent actually
      answered with domain knowledge rather than a generic response.
    """
    if isinstance(outputs, dict):
        _response = outputs.get("content") or outputs.get("final_response") or str(outputs)
    else:
        _response = str(outputs) if outputs is not None else ""
    response_lower = _response.lower()
    expected_agent = (inputs or {}).get("expected_agent") or (expectations or {}).get("expected_agent")

    if expected_agent and expected_agent in _AGENT_SIGNALS:
        signals = _AGENT_SIGNALS[expected_agent]
        if expected_agent == "multi":
            domains = sum(1 for sigs in _AGENT_SIGNALS.values() if any(s in response_lower for s in sigs))
            return Feedback(value=min(1.0, domains / 4), rationale=f"{domains} domains detected")
        matched = [s for s in signals if s in response_lower]
        score = min(1.0, len(matched) / max(2, len(signals) // 2))
        return Feedback(value=score, rationale=f"Matched: {matched}" if matched else "No expected-domain signals found")

    # Production path: score by best-matching domain
    best_domain, best_count = None, 0
    for domain, signals in _AGENT_SIGNALS.items():
        count = sum(1 for s in signals if s in response_lower)
        if count > best_count:
            best_domain, best_count = domain, count
    if best_count == 0:
        return Feedback(value=0.0, rationale="No domain-specific vocabulary found — generic or empty response")
    score = min(1.0, best_count / 3)
    return Feedback(value=score, rationale=f"Best domain: {best_domain} ({best_count} signals matched)")


@scorer
def cites_specific_data(inputs: dict, outputs, expectations: dict = None) -> Feedback:
    """Checks the response contains concrete data points (numbers, IDs, dates) not just hedges."""
    import re as _re
    if isinstance(outputs, dict):
        response = outputs.get("final_response") or outputs.get("content") or str(outputs)
    else:
        response = str(outputs) if outputs is not None else ""
    if not response or response.startswith("[ERROR:") or response.startswith("[NO RESPONSE"):
        return Feedback(value=0.0, rationale=f"No usable response: {response[:60]!r}")
    patterns = [
        r'\d+\.?\d*\s*%',
        r'\$\s*[\d,]+',
        r'[£€]\s*[\d,]+',
        r'\b[\d,]+(?:\.\d+)?\s*(?:USD|GBP|EUR|K|M)\b',
        r'(?:CK|AUD|REG|INS|CON)-[\w-]+',
        r'\d{4}-\d{2}-\d{2}',
        r'\b(?:score|grade|rating)\s*(?:of\s*)?[\d.]+',
        r'\b[\d.]+x\b',
        r'\b\d+\s+(?:cases?|orders?|locations?|findings?|violations?|reports?)',
        r'\b\d{3,}(?:,\d{3})*\b',
    ]
    found = [p for p in patterns if _re.search(p, response, _re.IGNORECASE)]
    hedges = ["i don't have access", "i cannot provide", "i'm unable", "consult your", "please check with"]
    is_hedged = any(h in response.lower() for h in hedges)
    if is_hedged and not found:
        return Feedback(value=0.0, rationale="Hedged response with no data — retrieval likely failed")
    return Feedback(value=min(1.0, len(found) / 3.0), rationale=f"{len(found)} data patterns found")


print("✅ Scorer definitions ready")


In [ ]:
_WANTED_SCORERS = {"safety", "relevance_to_query", "ceo_quality", "routing_accuracy", "cites_specific_data"}

# Remove any stale scorers from previous runs that are no longer wanted.
_all_existing = list_scorers(experiment_id=EXP_ID)
for _s in _all_existing:
    if _s.name not in _WANTED_SCORERS:
        try:
            _s.stop()
        except Exception:
            pass
        print(f"  🗑  Removed stale scorer: {_s.name}")

def _register_scorer(scorer_obj, name: str, sample_rate: float = 1.0):
    existing = {s.name: s for s in list_scorers(experiment_id=EXP_ID)}
    sampling = ScorerSamplingConfig(sample_rate=sample_rate)
    if name in existing:
        existing[name].start(sampling_config=sampling)
        print(f"  ↺ {name} — restarted at {sample_rate:.0%} sample rate")
        return existing[name]
    registered = scorer_obj.register(name=name, experiment_id=EXP_ID)
    registered.start(sampling_config=sampling)
    print(f"  ✅ {name} — registered + started at {sample_rate:.0%} sample rate")
    return registered

print(f"Registering scorers on: {EXP_NAME}\n")

_register_scorer(Safety(),           name="safety",             sample_rate=1.0)
_register_scorer(RelevanceToQuery(), name="relevance_to_query", sample_rate=1.0)
_register_scorer(
    Guidelines(
        guidelines=(
            "The response must include specific data points such as numbers, percentages, "
            "dates, case numbers, or named locations. "
            "The response must not be a generic hedge or refusal (e.g. 'I don't have access'). "
            "The response must directly answer the question asked."
        ),
    ),
    name="ceo_quality",
    sample_rate=1.0,
)
_register_scorer(routing_accuracy,   name="routing_accuracy",   sample_rate=1.0)
_register_scorer(cites_specific_data, name="cites_specific_data", sample_rate=1.0)

print(f"\n✅ Scorers active on {EXP_NAME}")
print("   Scores will appear in the Assessments column for every new trace.")


In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()

print("=" * 60)
print("CEO Supervisor — MLflow Setup Summary")
print("=" * 60)
print(f"\n📦 Eval dataset (UC):  {UC_DATASET_TABLE}")
print(f"   Load with:  eval_dataset = mlflow.genai.datasets.get_dataset('{UC_DATASET_TABLE}')")
print(f"\n📊 Experiment:  {host}/ml/experiments/{EXP_ID}")

registered = list_scorers(experiment_id=EXP_ID)
print(f"\n🔍 Scorers ({len(registered)} registered):")
for s in registered:
    state = getattr(s, 'state', 'unknown')
    rate  = getattr(getattr(s, 'sampling_config', None), 'sample_rate', '?')
    print(f"   • {s.name:<25} state={state}  sample_rate={rate}")
